# FACTS Multimodal Benchmark - Starter Code

The [FACTS Multimodal Benchmark](https://www.kaggle.com/benchmarks/google/facts-multimodal) is designed to evaluate the factual accuracy, faithfulness, and completeness of AI models when responding to prompts that include **images**.

This notebook demonstrates the complete evaluation pipeline:

1) **Setting up**: Installing dependencies and defining Protocol Buffer (protobuf) specifications.
2) **Data Loading**: Downloading the dataset and pre-processing the multimodal inputs (image and rubrics).
3) **Evaluation Logic**: Defining helper functions for factuality and quality scoring using judge models.
4) **Task Definition**: Implementing the core kbench.task function.
5) **Execution & Aggregation**: Running the benchmark and calculating the final mean score with a confidence interval.

## Setup and Dependencies

Install the necessary benchmark utilities and the grpcio-tools package for compiling the protobuf file.

In [ ]:
!git clone https://github.com/Kaggle/kaggle-benchmarks.git
!cd kaggle-benchmarks
!uv sync -q --project /benchmarks --group research_benchmarks


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
error: No `pyproject.toml` found in current directory or any parent directory


In [ ]:
!pip install -q hishel grpcio-tools httpx openai google-genai 

In [1]:
import sys
sys.path.insert(0, "/home/work/yuna/HPA/eda/kaggle-benchmarks/src")
import pandas as pd
import base64
import json
import numpy as np
from pathlib import Path 
from google.protobuf import text_format

# Import core kaggle_benchmarks and utility libraries
import kaggle_benchmarks as kbench
from kaggle_benchmarks import content_types
from kaggle_benchmarks.kaggle import serialization
import math
from scipy.stats import norm
from datetime import datetime
import kagglehub

# Initialize the Kaggle Client
kbench.client = kbench.kaggle.KaggleClient(use_cache=True)

## Define Rubric Protobuf
The benchmark uses a Protocol Buffer (.proto file) to define the structure for the ground-truth "rubrics" used by the judge models. This cell writes the definition and compiles it into a Python module.

In [ ]:
%%writefile /kaggle/working/superhuman_rating_instructions.proto
syntax = "proto3";

// Rating rubric items
message RubricItem {
  oneof value {
    // For static prompts, the rubric contains a textual fact.
    string fact_text = 1;
    // For dynamic prompts, the rubric contains a URL to a webpage.
    string golden_url = 2;
  }
  // Tags for metadata related to the rubric item.
  repeated string tags = 3;
}

// Collection of rating rubric items
message RatingRubrics {
  repeated RubricItem rubric_items = 1;
}

In [ ]:
# Compile the .proto file into a Python module
!python -m grpc_tools.protoc \
    --proto_path=/kaggle/working/ \
    --python_out=/kaggle/working/ \
    superhuman_rating_instructions.proto

# Import the compiled module
sys.path.append("/kaggle/working")
import superhuman_rating_instructions_pb2

## Configuration and Constants
Define model handles, dataset constants, and column names.

In [2]:
# --- Judge Model Configuration ---
FACTUALITY_JUDGES = ["openai/o3-2025-04-16", "google/gemini-2.5-pro"]
QUALITY_JUDGE = "google/gemini-2.5-flash"

# --- Constants for Data Loading ---
KAGGLEHUB_HANDLE = "deepmind/facts-multimodal-v2-public-data"
DATASET_FILE = "facts_multimodal_public.csv"

## Define Prompts


This benchmark requires two main judge prompts: one for **Quality/Coverage** evaluation and one for **Factuality/Contradiction** evaluation.
Since the prompts are lengthy and detailed, we use templates that the judge logic fills with the specific `RESPONSE`, `FACT_LIST`, and `INPUT_IMAGE`.

1) **Quality Template**: Assesses **coverage** (entailment) of the response against essential facts in the rubric.
2) **Factuality Prompt Template**: Checks for **contradictions** against the image, fact list, and global truth.

In [ ]:
QUALITY_PROMPT_TEMPLATE = """## TASK DESCRIPTION

**You are an AI evaluator specializing in response entailment. Your task is to check the coverage of a machine-generated response (`RESPONSE`) against a provided list of facts (`IMAGE_FACT_LIST`).** Your goal is to determine, for **EACH SENTENCE** in the response, whether its core information is **covered, mentioned, or logically entailed** by the `IMAGE_FACT_LIST`.

You will not be checking the `RESPONSE` for factual errors. **Coverage is determined *only* by the text written in the `RESPONSE`.** Information that is visible in the `INPUT_IMAGE` but is absent from the `RESPONSE` text does **not** count as covered.

---

## INPUTS

*   **`INPUT_PROMPT`**: The user's prompt that generated the response.
*   **`INPUT_IMAGE`**: The image the response is about (used for context if needed to understand the fact).
*   **`IMAGE_FACT_LIST`**: A list of facts about the `INPUT_IMAGE` to be checked for coverage.
*   **`RESPONSE`**: The machine-generated response to be evaluated for coverage.

---

## UNDERSTANDING THE `IMAGE_FACT_LIST`

`IMAGE_FACT_LIST` is formatted as:
`Fact # <ID> Id </ID> <FACT> Fact </FACT> <TYPE> Type </TYPE> <IMPORTANCE> Importance </IMPORTANCE>`

Where:
*   `Id`: Fact number.
*   `FACT`: The statement of fact.
*   `TYPE`: Defines how to interpret the fact (e.g., `Sentence`, `Exhaustive set`).

---

## EVALUATION PROCESS (Per Fact in `IMAGE_FACT_LIST`)

For **EACH FACT** in the `IMAGE_FACT_LIST`, perform the following steps:

1.  **Analyze the Fact:** Carefully read the fact statement and identify its core informational content. What is the key piece of information that must be present in the `RESPONSE` for this fact to be considered covered?

2.  **Scan the `RESPONSE`:** Thoroughly read the entire `RESPONSE` to locate any text that relates to the fact's core information.

3.  **Judge Coverage ("Yes" or "No"):**
    *   Assign **"Yes"** if the fact is covered. A fact is considered covered if:
        *   **Direct Mention:** The core information is stated in the `RESPONSE`. The wording does not need to be identical, but the meaning must be the same.
        *   **Logical Entailment:** The fact can be directly and unambiguously inferred from one or more statements in the `RESPONSE`. The inference must be simple and not require large logical leaps.
        *   **Sufficient Detail for Sets:** For facts of type `Exhaustive set` or `Non-exhaustive set`, the key information about the set (e.g., all its specific members mentioned in the fact, or its total count) is present in the `RESPONSE`. For example, if the fact is "The animals are a dog and a cat (Exhaustive)", the response must mention both a dog and a cat to be "Yes". If it only mentions a dog, the coverage is "No".

    *   Assign **"No"** if the fact is not covered. A fact is not covered if:
        *   **Complete Absence:** The core information of the fact is entirely missing from the `RESPONSE`.
        *   **Insufficient Detail:** The `RESPONSE` mentions a related topic but omits the specific details that constitute the core of the fact. For example, if the fact is "The man is wearing a red hat" and the response only says "A man is in the picture," the fact is not covered.

    > **CRITICAL REMINDER:** Your decision must be based **solely on the text content of the `RESPONSE`**. Do not mark a fact as "Yes" just because it is true according to the `INPUT_IMAGE`. The information must be explicitly written or logically entailed by the `RESPONSE` text.
    
4. **Review the Judgetment:**
    *   If the judgement is "Yes", double check that the fact is covered by the `RESPONSE`.
    *   If the judgement is "No", double check that the fact is not covered by the `RESPONSE`. Do not penalize shorter responses if they contain the core information from the fact that answers the prompt.
        * If the prompt is "What is the color of the shirt?", and the fact says "The man's shirt is black" and the response says "black", the fact is covered.

4.  **Repeat:** Repeat steps 1-3 for every fact in the `IMAGE_FACT_LIST`.

---

## OUTPUT FORMAT

Your entire output **must be a single, valid JSON object.** Do not include any other text, comments, or explanations before or after the JSON code block.

The JSON object must contain key-value pairs where:
*   The **key** is a string that corresponds to the fact's `Id` from the input. It should be formatted as `"Fact <ID>"`, where `<ID>` is the number from the fact list (e.g., the fact with `<ID>1</ID>` becomes the key `"Fact 1"`, `<ID>2</ID>` becomes `"Fact 2"`, etc.).
*   The **value** is a string, either **"Yes"** or **"No"**, indicating if that fact is covered by the `RESPONSE`.
After all of the facts, must also output "Proportion of Facts Covered" which is the number of facts with a value of "Yes" divided by the total number of facts.

**Example:**
```json
{
  "Fact 1": "Yes",
  "Fact 2": "Yes",
  "Fact 3": "No",
  "Fact 4": "Yes",
  "Proportion of Facts Covered": 0.75
}
```


# Your task
## User query
<|begin_of_query|>
{{user_request}}
<|end_of_query|>

## Test Response:
<|begin_of_test_response|>
{{test_response}}
<|end_of_test_response|>

## Fact List:
<|begin_of_fact_list|>
{{rubrics}}
<|end_of_fact_list|>

Please write your analysis and final verdict for the test response.""".strip()



FACTUALITY_PROMPT_TEMPLATE = """
SPECIAL INSTRUCTION: think silently if needed.

## TASK DESCRIPTION:

**You are a highly meticulous fact-checking expert tasked with verifying the factual accuracy of a machine-generated response (denoted by `RESPONSE`) against multiple sources of truth: a provided fact list (denoted by `IMAGE_FACT_LIST`), the provided image itself (denoted by `INPUT_IMAGE`), and established global truth or widely accepted web knowledge.** Your goal is to identify *any* instances where the `RESPONSE` *contradicts*, *misrepresents*, is *inconsistent with*, or makes factually incorrect claims when compared against *any* of these truth sources.

**You must be extremely strict and aim for very high recall of factual inaccuracies.** Even a tiny mismatch or inconsistency should be flagged. The response does *not* need to include all possible facts; your focus is solely on the accuracy of the claims the `RESPONSE` *does* make.

**INPUTS:**

*   **`INPUT_PROMPT`**: The user's prompt that generated the response.
*   **`INPUT_IMAGE`**: The image the response is about.
*   **`IMAGE_FACT_LIST`**: A list of facts about the `INPUT_IMAGE` to aid fact-checking.
*   **`RESPONSE`**: The machine-generated response to be evaluated.

**UNDERSTANDING THE `IMAGE_FACT_LIST`:**

`IMAGE_FACT_LIST` is formatted as:
`Fact# <ID> Id </ID> <FACT> Fact </FACT> <TYPE> Type </TYPE> <IMPORTANCE> Importance </IMPORTANCE>`

Where:
*   `Id`: Fact number.
*   `FACT`: The statement of fact.
*   `TYPE`: Defines how to interpret the fact:
    *   **Sentence**: Treat as an absolute truth. `RESPONSE` must not contradict any part of this statement.
    *   **Exhaustive set**: Defines a complete category. If `RESPONSE` mentions items from this category, they *must* be present in the list. Any claims about the *number* or *completeness* of items in this category must be accurate according to the list.
    *   **Non-exhaustive set**: Defines an incomplete category. `RESPONSE` must not contradict the listed items' membership in the category. However, the list is not complete, so the absence of an item from the response that *is* in the image/global truth but *not* on this list is *not* a contradiction *based on this list alone* (but might be a contradiction based on the image or global truth).
*   `IMPORTANCE`: Indicates relevance ("essential" or "optional") to answering `INPUT_PROMPT`, but *does not affect factuality checking*. All facts in the list are considered true for evaluation.

**EVALUATION PROCESS (Per Sentence in `RESPONSE`):**

For **EACH SENTENCE** in `RESPONSE`, perform the following steps rigorously:

1.  **Identify ALL Claims:** Carefully identify *all* explicit and implicit claims made in the sentence. This includes:
    *   **Explicit Statements:** Directly stated facts, assertions, quantities, names, descriptions, relationships, etc.
    *   **Implicit Statements:** Meanings, interpretations, tones, or judgments *reasonably implied* by the sentence in the context of the prompt and image.
    *   **Calculations/Reasoning:** Any presented mathematical calculations or logical deductions.

2.  **Gather and Verify Evidence:** Check each claim against **ALL** relevant sources:
    *   **`IMAGE_FACT_LIST`**: Search for facts that *directly support* or *contradict* the claim, respecting the `TYPE` rules (Sentence, Exhaustive, Non-exhaustive).
    *   **`INPUT_IMAGE`**: *Visually inspect* the image for evidence that directly supports or contradicts the claim. Look for objects, text, relationships, appearances, colors, counts, actions, etc.
    *   **Global Truth / Web Knowledge**: *Evaluate the claim against* established, widely accepted facts (e.g., historical dates, scientific facts, common knowledge, information readily verifiable through reliable web search).

3.  **Detect Disagreements (Contradictions):** A claim constitutes a **CONTRADICTION** if it meets **ANY** of the following criteria based on *any* of the evidence sources (`IMAGE_FACT_LIST`, `INPUT_IMAGE`, Global Truth):
    *   **Direct Contradiction:** The claim asserts something that is the *direct opposite* of, or clearly *incompatible with*, information from the fact list, the image contents, or established global truth.
    *   **Fact List Violation:** The claim violates the rules for "Exhaustive set" or "Non-exhaustive set" facts as defined above.
    *   **Visual Inconsistency:** The claim describes something that is *demonstrably false* based on the visual evidence in `INPUT_IMAGE`.
    *   **Global Truth Inaccuracy:** The claim *contradicts well-established* real-world facts or common knowledge.
    *   **Misrepresentation:** The claim *distorts*, *misrepresents*, or *inaccurately characterises* information from the fact list, the image, or global truth, even if not a direct opposite.
    *   **Incorrect Calculation/Reasoning:** Any presented math or logic is *flawed or leads to an incorrect result* based on the premises (which themselves must be factual).

    **Important Considerations:**
    *   **Strictness:** Be extremely strict. If there is *any doubt* or *minor mismatch*, lean towards identifying a contradiction. **High recall** of errors is paramount.
    *   **Completeness vs. Accuracy:** The `RESPONSE` does not need to be exhaustive. Omissions are *not* contradictions unless the `RESPONSE` makes a claim of completeness that is false (e.g., "There are only two people shown" when there are three) or violates an "Exhaustive set" rule.
    *   **Additional Details:** Information in `RESPONSE` that *expands upon* the provided facts *without contradicting* any source is **NOT** a contradiction *if it is factually accurate* according to the image or global truth. However, if such additional detail *is* inaccurate, it **IS** a contradiction.
    *   **Consistency:** Claims must be internally consistent within the `RESPONSE` and externally consistent with all truth sources.
    *   **Ignore Citations:** Citations like `[1]`, `[2]` are **NOT** part of the `RESPONSE` content and should be ignored during fact-checking.

4.  **Repeat:** Repeat steps 1-3 for *every sentence* in `RESPONSE`.

**FINAL VERDICT:**

*   **HAS CLEAR CONTRADICTION(s).**: If *any* contradiction (as defined in step 3) is found in *any* sentence based on *any* truth source (`IMAGE_FACT_LIST`, `INPUT_IMAGE`, Global Truth).
*   **NO CLEAR CONTRADICTION.**: If *no* contradictions are found after meticulously checking every claim in every sentence against all truth sources.

**OUTPUT FORMAT:**

**final verdict:** <Either **HAS CLEAR CONTRADICTION(s).** or **NO CLEAR CONTRADICTION.**>
**reasoning:** <A detailed step-by-step explanation for the verdict. For *each* identified contradiction, clearly state the specific claim in the `RESPONSE`, the source of truth it contradicts (`IMAGE_FACT_LIST` fact ID, `INPUT_IMAGE` detail, or Global Truth), and why it is a contradiction. If there are no contradictions, briefly explain why the response aligns with the facts/image/global truth.>
*   **contradicting sentences:** <List *all* sentences from `RESPONSE` that contain *at least one* contradiction. Each sentence should start on a new line marked with a bullet point (e.g., `- Sentence text...`). If no contradictions, state "None".>
*   **There are** X **contradicting sentences found in `RESPONSE`.** (Where X is the total count of sentences listed above. X should be 0 if there are no contradictions).
 """

## Helper Functions for Parsing and Formatting
These functions handle the conversion of the raw data (rubrics) into a usable format and parse the structured output from the judge models.

In [ ]:
def _parse_factuality(response: str) -> bool:
    """Parses the factuality judge's response."""
    if "HAS CLEAR CONTRADICTION" in response:
        return False
    elif "NO CLEAR CONTRADICTION" in response:
        return True
    else:
        return None


def _parse_quality_answer_float_json(ans) -> tuple[float, float]:
    """Extracts the calculated fact score and the proportion covered from judge JSON output."""
    output = json.loads(ans)
    
    yes_count = 0
    no_count = 0
    for fact in output:
        if fact[:4].lower() != "fact":
            continue
        if output[fact].lower() == "yes":
            yes_count += 1
        elif output[fact].lower() == "no":
            no_count += 1
        else:
            continue
        
    total_count = yes_count + no_count
    calculated_score = yes_count / (total_count) if total_count > 0 else np.nan
    
    proportion_covered = float(output.get("Proportion of Facts Covered", np.nan))
    
    return calculated_score, proportion_covered


def _parse_quality(ans):
    """Robustly parses the quality judge response to extract the supplied score."""
    try:
        _, supplied_score = _parse_quality_answer_float_json(ans)
        return supplied_score
    except:
        if "```json" in ans:
            ans = ans.split("```json")[1]
            ans = ans.split("```")[0]
        # Attempt to extract JSON block if it's just wrapped in braces
        elif ans.find("{") != -1 and ans.find("}") != -1:
            ans = ans[ans.find("{") :]
            ans = ans[: ans.find("}") + 1]
        else:
            return np.nan
        
        try:
            _, supplied = _parse_quality_answer_float_json(ans)
        except:
            return np.nan
        return supplied


### Rubric Formatters

def get_rubrics_proto(text_formatted_string):
    """Parses a text-formatted string into a RatingRubrics protobuf."""
    text_formatted_string = text_formatted_string.strip()
    if not text_formatted_string or text_formatted_string == "[]":
        return None
    try:
        return text_format.Parse(text_formatted_string, superhuman_rating_instructions_pb2.RatingRubrics())
    except Exception as e:
        print(f"Error parsing rubrics: {e}\n{text_formatted_string}")
        return None


def get_rubrics_string(rubrics):
    """Formats rubrics for the factuality prompt with XML-like tags."""
    rubric_str = ""
    if rubrics is None:
        return rubric_str

    for j, rubric_item in enumerate(rubrics.rubric_items):
        rubric_str += (
            f"Fact # <ID> {j + 1} </ID>"
            f" <FACT> {rubric_item.fact_text} </FACT>"
            f" <TYPE> {rubric_item.tags[0]} </TYPE>"
            f" <IMPORTANCE> {rubric_item.tags[1]} </IMPORTANCE>\n"
        )
    return rubric_str

def get_quality_rubrics_string(rubrics):
    """Formats rubrics for the quality prompt, filtering for only essential facts."""
    rubric_str = ""
    if rubrics is None:
        return rubric_str
    
    for j, item in enumerate(rubrics.rubric_items):
        if (
            item.tags[1] == "optional"
            or item.tags[1] == "non-essential"
        ):
            continue
        
        rubric_str += (
            f"Fact # <ID> {j + 1} </ID>"
            f" <FACT> {item.fact_text} </FACT>"
            f" <TYPE> {item.tags[0]} </TYPE>\n"
        )
    return rubric_str

## Data Loading and Preprocessing
Download the dataset, convert the rubrics from a string format into the Protocol Buffer object, and decode the base64 image strings into raw bytes.

In [ ]:
from IPython.display import Image, display
import pprint

row = facts_df.iloc[0].to_dict()
display(Image(url=row['img_url'], width=500)) 
row 

In [12]:
row

{'prompt': 'Write an elaborative description for the image.',
 'item_id': 12372907390863178199,
 'rubrics': 'rubric_items {\n  fact_text: "The image shows a blue sedan parked on the side of an empty road with the passenger side of the car facing towards the viewer."\n  tags: "Sentence"\n  tags: "essential"\n  tags: "From user-provided context"\n  tags: " "\n}\nrubric_items {\n  fact_text: "In the image, the sedan is underneath a large tree and is the most prominent element in the foreground of the image."\n  tags: "Sentence"\n  tags: "essential"\n  tags: "From user-provided context"\n  tags: " "\n}\nrubric_items {\n  fact_text: "In the image, there is a large, leafy tree with a brown trunk that is planted on a green, grassy lawn."\n  tags: "Sentence"\n  tags: "essential"\n  tags: "From user-provided context"\n  tags: " "\n}\nrubric_items {\n  fact_text: "In the image, behind the tree and the car is a multi-story parking garage."\n  tags: "Sentence"\n  tags: "essential"\n  tags: "From u

In [8]:
# Download the dataset from KaggleHub
dataset_dir = Path(kagglehub.dataset_download(KAGGLEHUB_HANDLE))
facts_df = pd.read_csv(dataset_dir / DATASET_FILE)

# Explicitly select the required columns for the task
# facts_df = facts_df[['prompt', 'img_bytes', 'mime_type', 'rubrics']]

# Convert the rubric strings into protobuf objects
# facts_df["rubrics"] = facts_df["rubrics"].apply(get_rubrics_proto)
# facts_df["img_bytes"] = facts_df["img_bytes"].apply(base64.b64decode)
facts_df

,prompt,item_id,rubrics,user_intent_majority,external_information_majority,reasoning_requirement_majority,img_url,prompt_category,image_category
0,Write an elaborative description for the image.,12372907390863178199,"rubric_items {\n fact_text: ""The image shows ...",no_majority,"Yes, the user explicitly or implicitly require...",No,https://storage.googleapis.com/docci/data/imag...,Visual Description & Captioning,Architecture & Building Exteriors
1,Write an elaborative description for the image.,3500076804783627904,"rubric_items {\n fact_text: ""In the image, th...",Creative writing,no_majority,No,https://vader-prod.s3.amazonaws.com/1709655668...,Visual Description & Captioning,Product & E-commerce Imagery
2,What all is happening in this image? What are ...,4868913978793301692,"rubric_items {\n fact_text: ""There are seven ...",Input understanding or processing,"Yes, the user explicitly or implicitly require...",No,https://imagenesdereflexion.org/wp-content/upl...,Visual Description & Captioning,"People, Portraits & Events"
3,Where can you find this bird?,15201862648355542096,"rubric_items {\n fact_text: ""Chrysuronia goud...",External information seeking,"No, it is impossible to only use information f...",No,https://upload.wikimedia.org/wikipedia/commons...,Reasoning,Nature & Wildlife
4,What historical event is shown in the picture?,220102809251999825,"rubric_items {\n fact_text: ""The image is a b...",External information seeking,"No, it is impossible to only use information f...",No,https://media.licdn.com/dms/image/v2/D5612AQEZ...,Visual Description & Captioning,"People, Portraits & Events"
...,...,...,...,...,...,...,...,...,...
706,What is the building in this picture?,13094474774033897133,"rubric_items {\n fact_text: ""The building in ...",External information seeking,no_majority,No,http://ocdn.eu/images/pulscms/NTI7MDA_/c8ce0cd...,Object & Entity Recognition/Identification,Architecture & Building Exteriors
707,"What is the name of this traditional garment, ...",1421269499063353808,"rubric_items {\n fact_text: ""The garment pict...",External information seeking,"No, it is impossible to only use information f...",No,https://newscast.jp/attachments/TooaSImMtcPaRY...,Abstract Concept & Cultural Understanding,"People, Portraits & Events"
708,Write an elaborative description for the image.,16895566352749273991,"rubric_items {\n fact_text: ""The user-provide...",Creative writing,no_majority,No,https://storage.googleapis.com/docci/data/imag...,Visual Description & Captioning,Transportation & Vehicle
709,Write an elaborative description for the image.,2689297171271947972,"rubric_items {\n fact_text: ""The image shows ...",Input understanding or processing,"Yes, the user explicitly or implicitly require...",No,https://storage.googleapis.com/docci/data/imag...,Visual Description & Captioning,Nature & Wildlife


## Judge Logic Functions
These functions define how the judge models are called to evaluate the quality and factuality of the evaluated model's response.

In [ ]:
def determine_factuality(judge_model: str, row: dict, response: str) -> bool:
    """
    Runs the factuality check using a judge model.
    The check is run twice and returns True if at least one run passes.
    """
    rubric_str = get_rubrics_string(row["rubrics"])
    
    system_instructions = FACTUALITY_PROMPT_TEMPLATE
    
    # Text content sent BEFORE the image
    pre_image_text = f"""
<IMAGE_FACT_LIST>
{rubric_str}
</IMAGE_FACT_LIST>
<INPUT_IMAGE>
  """.strip()

    # Text content sent AFTER the image
    post_image_text = f"""
</INPUT_IMAGE>
<INPUT_PROMPT>
{row['prompt']}
</INPUT_PROMPT>
<RESPONSE>
{response}
</RESPONSE>

  """.strip()
    
    encoded_image = base64.b64encode(row['img_bytes']).decode('utf-8')
    image_content = content_types.images.from_base64(encoded_image, format=row['mime_type'].split('/')[-1])

    bool_responses = []
    for i in range(2): 
        with kbench.chats.new(name=f"factuality_judge_{judge_model.replace('/', '_')}", system_instructions=system_instructions):
            try:
                kbench.user.send(pre_image_text)
                kbench.user.send(image_content)
                judge_response = kbench.llms[judge_model].prompt(
                    post_image_text,
                    seed=42 + i
                )
                bool_responses.append(_parse_factuality_v2(judge_response))
            except Exception as e:
                print(f"Error getting factuality judge '{judge_model}' response: {e}")
                bool_responses.append(None)

    return any(r for r in bool_responses if r is not None)

### Quality Judge
def determine_quality_v2(row: dict, response: str) -> float:
    """Runs the V2 quality check in an isolated chat context."""
    rubric_str = get_quality_rubrics_string_v2(row["rubrics"])
    
    if not rubric_str: return 1.0

    prompt = QUALITY_PROMPT_TEMPLATE.replace("{{user_request}}", row["prompt"])
    prompt = prompt.replace("{{test_response}}", response)
    prompt = prompt.replace("{{rubrics}}", rubric_str)
    
    with kbench.chats.new(name=f"quality_judge"):
        try:
            judge_response = kbench.llms[QUALITY_JUDGE].prompt(prompt)
            quality_score = _parse_quality_v2(judge_response)
            return quality_score if not np.isnan(quality_score) else 0.0
        except Exception as e:
            print(f"Error getting quality judge response: {e}")
            return 0.0

## Define the Benchmark Task
The core task function, decorated with `@kbench.task`, runs the full cycle: model response generation, quality check, and factuality check.

In [ ]:
@kbench.task(name="facts_multimodal_task")
def facts_multimodal_task(llm, prompt: str, img_bytes: bytes, mime_type: str, rubrics) -> dict:
    """
    Generates a response for a multimodal prompt and evaluates its factuality and quality.
    """
    response = ""
    try:
        encoded_image = base64.b64encode(img_bytes).decode('utf-8')
        image_content = content_types.images.from_base64(encoded_image, format=mime_type.split('/')[-1])
        
        kbench.user.send(image_content)
        response = llm.prompt(prompt)
    except Exception as e:
        print(f"Error getting model response: {e}")
        response = "" 


    row_dict = {
        'prompt': prompt, 'img_bytes': img_bytes,
        'mime_type': mime_type, 'rubrics': rubrics
    }

    quality_score = determine_quality(row_dict, response)


    factuality_pass_per_judge = [
        determine_factuality(judge, row_dict, response) for judge in FACTUALITY_JUDGES
    ]
    item_factuality_pass = any(factuality_pass_per_judge)

    item_pass = (item_factuality_pass is True) and (quality_score > 0.5)
    final_score = 1.0 if item_pass else 0.0

    return {
        "response": response,
        "item_factuality_pass": item_factuality_pass,
        "item_quality_score": quality_score,
        "final_score": final_score,
    }

## Run the Benchmark
Execute the defined task across the dataset, handling parallel execution and retries.

In [ ]:
# Evaluate the task on the dataset
runs = facts_multimodal_task.evaluate(
    llm=[kbench.llms['google/gemini-2.5-pro']],
    evaluation_data=facts_df,
    n_jobs=10,
)

run_files = serialization.get_runs_filenames(runs)

## Aggregate and Finalize Results
Define the aggregation function to calculate the mean score and confidence interval, then merge the individual run results.

In [ ]:
def run_results_to_df(run_results: list[dict], columns: list[str]) -> pd.DataFrame:
    """Converts a list of run result dictionaries to a pandas DataFrame."""
    data = []
    for result in run_results:
        if "dictResult" in result:
            row = {col: result["dictResult"].get(col) for col in columns}
            data.append(row)
    return pd.DataFrame(data)

def get_confidence_radius(success_cnt: int, sample_cnt: int, confidence_level=0.95) -> float:
    """
    Calculates the margin of error (CI radius) for a binomial proportion 
    using the Normal Approximation method (for pass/fail metrics).
    """
    if sample_cnt == 0:
        return 0.0
    
    assert success_cnt >= 0
    assert sample_cnt > 0
    assert success_cnt <= sample_cnt

    p_hat = success_cnt / sample_cnt
    assert p_hat >= 0.0
    assert p_hat <= 1.0

    z = norm.ppf(1 - (1 - confidence_level) / 2)

    margin_of_error = z * math.sqrt((p_hat * (1 - p_hat)) / sample_cnt)
    return margin_of_error


def get_confidence_radius_from_df(df: pd.DataFrame, column: str) -> float:
    """Calculates the CI radius for a binary column (0.0 or 1.0)."""
    success_cnt = df[column].sum()
    sample_cnt = len(df)
    return get_confidence_radius(success_cnt=success_cnt, sample_cnt=sample_cnt)


def calculate_metrics(run_results: list[dict]) -> tuple[float, float]: # Changed return type hint
    """Calculates the mean final score and its 95% confidence interval, returning a tuple."""
    df = run_results_to_df(run_results, columns=["final_score"])

    mean_score = df["final_score"].mean()
    ci = get_confidence_radius_from_df(df, "final_score")
    
    return (float(mean_score), float(ci))

def calculate_coverage(df: pd.DataFrame) -> float:
    return df["item_quality_score"].mean()

def calculate_factuality(df: pd.DataFrame) -> float:
    return df["item_factuality_pass"].mean()

# Merge and aggregate the results from the run files
final_result = serialization.merge_results_from_runfiles(
    run_files=run_files, 
    aggregate_fn=calculate_metrics, 
    delete_run_files=True 
)

print(f"Final aggregated result file: {final_result}")

## View Aggregated Results
Load the aggregated results file, parse the execution metadata, and display the final score and confidence interval in a readable summary table.

In [ ]:
# The aggregated file is saved with the task name.
aggregated_filename = 'facts_multimodal_task-Run_aggregated.run.json'

# Load and parse the aggregated results
with open(aggregated_filename, 'r') as f:
    results = json.load(f)

# Extract key information
model_version = results.get('modelVersion', {}).get('slug', 'N/A')
start_time = results.get('startTime', 'N/A')
end_time = results.get('endTime', 'N/A')

# Parse timestamps for better readability
if start_time != 'N/A':
    start_time = datetime.fromisoformat(start_time.replace('Z', '+00:00')).strftime('%Y-%m-%d %H:%M:%S UTC')
if end_time != 'N/A':
    end_time = datetime.fromisoformat(end_time.replace('Z', '+00:00')).strftime('%Y-%m-%d %H:%M:%S UTC')

# Extract numeric result (mean score and CI radius)
mean_score = 'N/A'
ci_radius = 'N/A'
for result in results.get('results', []):
    if 'numericResult' in result:
        # 1. Get the numeric result object
        num_result = result['numericResult']
        
        # 2. Extract the mean score from the 'value' field
        mean_score = num_result.get('value', 'N/A')
        
        # 3. Extract the CI radius directly from the 'confidenceInterval' field (it is a float)
        # Note: We use .get(..., 'N/A') on num_result to handle missing keys gracefully.
        ci_radius = num_result.get('confidenceInterval', 'N/A')
        break

# Format the final score string (Score ± CI)
final_score_str = f"{mean_score:.4f} ± {ci_radius:.4f}" if isinstance(mean_score, float) and isinstance(ci_radius, float) else "N/A"

# Create a summary dataframe
summary_df = pd.DataFrame([{
    'Model': model_version,
    'Start Time': start_time,
    'End Time': end_time,
    'Score (Mean ± CI)': final_score_str
}])

print("\n" + "="*80)
print("FACTS MULTIMODAL BENCHMARK RESULTS")
print("="*80)
display(summary_df)